# PROJECT KIRA — PHASE 2 RESEARCH EXPANSION
**Authoritative Execution Orchestrator**
- Baseline Run ID: `run_tiny_s20260827_193f7897_40997ab`
- Execution Backend: `CPU (NumPy vectorized)`

In [ ]:
import os
import sys
import time
import subprocess
import logging
from pathlib import Path
import json

# Ensure repo environment in Kaggle
if not os.path.exists("src"):
    if os.path.exists("/kaggle/working/Project-KIRA"):
        os.chdir("/kaggle/working/Project-KIRA")
    elif os.path.exists("/kaggle/input/project-kira"):
        os.chdir("/kaggle/input/project-kira")
    else:
        print("Cloning latest Project-KIRA repository...")
        subprocess.run(["git", "clone", "https://github.com/ankit-choubey/Project-KIRA.git", "/kaggle/working/Project-KIRA"], check=True)
        os.chdir("/kaggle/working/Project-KIRA")

if "src" not in sys.path:
    sys.path.insert(0, "src")

# Install lightweight requirements if missing
!pip install polars lightgbm scikit-learn > /dev/null 2>&1


In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

from mcdl.research.phase2.state import CheckpointManager
from mcdl.research.phase2 import experiments as exp

BASELINE_RUN = "run_tiny_s20260827_193f7897_40997ab"
BASELINE_COMMIT = "40997ab"

try:
    CURRENT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
except Exception:
    CURRENT_COMMIT = "bc090396d38f665851711af74b0b117bf5eb0984"

manager = CheckpointManager(
    run_id=f"phase2_{int(time.time())}",
    git_commit=CURRENT_COMMIT,
    baseline_run_id=BASELINE_RUN,
    baseline_git_commit=BASELINE_COMMIT
)
logger.info(f"Phase 2 Orchestrator Initialized (commit={CURRENT_COMMIT[:8]}).")


In [ ]:
# CPU Checkpoint 1: Base Integrity & Sensitivity
exp.run_s00(manager)
exp.run_s01(manager)
exp.run_a01(manager)
exp.run_a02(manager)
logger.info("CPU CHECKPOINT 1 COMPLETED.")


In [ ]:
# Relational Graph Experiments (G-01, G-02, G-04, G-05)
exp.run_g01(manager)
exp.run_g02(manager)
exp.run_g04(manager)
exp.run_g05(manager)
logger.info("RELATIONAL GRAPH STAGES COMPLETED.")


In [ ]:
# Conditional Graph Stages (G-03 Fusion)
state_g01 = manager.get_state("G01")
if state_g01 == "COMPLETED":
    exp.run_g03(manager)
else:
    logger.info("G-01 not completed; skipping G-03.")


In [ ]:
# Time-Gated Stages (R-01 RL, LLM-01 Planner)
time_elapsed = time.monotonic() - manager.global_start
time_remaining = (5.5 * 3600) - time_elapsed

if time_remaining > 45 * 60:
    exp.run_r01(manager)
    exp.run_llm01(manager)
else:
    logger.warning(f"SKIPPED_TIME_BUDGET: Only {time_remaining/60:.1f} mins remain before soft stop.")


In [ ]:
# S-02: Full-Scale KIRA Synthetic World Validation
exp.run_s02(manager)
logger.info("S-02 FULL-SCALE VALIDATION COMPLETED.")


In [ ]:
# S-03: Distribution Shift / Zero-Day Robustness
exp.run_s03(manager)
logger.info("S-03 ZERO-DAY ROBUSTNESS COMPLETED.")


In [ ]:
# Final Synthesis & Evidence Package
exp.run_final(manager)
logger.info("PHASE 2 MASTER EVIDENCE PACKAGE ASSEMBLED.")
